<a href="https://colab.research.google.com/github/noencrp87/labo3-2026ba/blob/main/316_AutoGluon.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AutoGluon
Matar una mosca con una  bazooka, entrena automaticamente modelos de:


*   Estadistica Clasica
*   Machine Learning ~ GBDT
*   Deep Learning


 ['SeasonalNaive', 'RecursiveTabular', 'DirectTabular', 'DynamicOptimizedTheta', 'Chronos2', 'Chronos2SmallFineTuned', 'AutoETS', 'ChronosWithRegressor[bolt_small]', 'TemporalFusionTransformer', 'DeepAR']



![Matar una mosca con una bazooka ](https://storage.googleapis.com/open-courses/austral2026-5da5/labo3/Bazooka_fly.jpg)

## 0.1 Init ambiente Google Colab

Conectar la virtual machine donde esta corriendo Google Colab con el  Google Drive, para poder tener persistencia de archivos

In [1]:
# primero establecer el Runtime de Python 3
from google.colab import drive
drive.mount('/content/.drive')

Mounted at /content/.drive


Para correr la siguiente celda es fundamental, POR UNICA VEZ, seguir los siguientes pasos

* Registrar usuario en Kaggle con la cuenta de email de la Universidad Austral
* Hacer el "Join Competition"  a la competencia de  Labo 3
* Generar el archivo kaggle.json  a partir de   https://www.kaggle.com/settings/account  y presione  "Create Legacy API Key"
* Crear carpeta  labo3  en  el Google Drive
* Dentro de la carpeta labo3 crear carpeta   kaggle
* Subir a la carpeta kaggle el archivo  kaggle.json


In [3]:
%%shell

mkdir -p "/content/.drive/MyDrive/Labo 3"
mkdir -p /content/buckets

rm -f /content/buckets/bl
ln -sfn "/content/.drive/MyDrive/Labo 3" /content/buckets/bl

mkdir -p ~/.kaggle
cp "/content/buckets/bl/kaggle/kaggle.json" ~/.kaggle/kaggle.json
chmod 600 ~/.kaggle/kaggle.json

mkdir -p /content/buckets/bl/exp
mkdir -p /content/buckets/bl/datasets
mkdir -p /content/datasets

# defino funcion descargar()
descargar() {
  carpeta_destino="/content/buckets/bl/datasets/" # Corrected from b1 to bl
  url_origen="https://storage.googleapis.com/open-courses/austral2026-5da5/labo 3/"
  archivo="$1"

  if ! test -f "$carpeta_destino""$archivo"; then
    wget  "$url_origen""$archivo"  -O "$carpeta_destino""$archivo"
  fi

  if ! test -f  "/content/datasets/""$archivo"; then
    cp  "$carpeta_destino""$archivo"  "/content/datasets/""$archivo"
  fi;
}


# hago la descarga efectiva, llamando a descargar()
descargar  "sell-in.txt.gz"
descargar  "tb_productos.txt"
descargar  "tb_stocks.txt"
descargar  "product_id_apredecir201912.txt"


# 1  Modelo AutoGluon

## 1.1 Init Experimento

In [4]:
# instalacion de paquetes que NO vienen por default en Colab
# autogluon es MUY pesado. varios minutos se perderan aqui
!pip install uv
!uv pip install -q kaggle
!uv pip install autogluon[all]


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.3/26.3 MB 53.1 MB/s eta 0:00:00
Using Python 3.12.13 environment at: /usr
Resolved 253 packages in 4.15s
Prepared 84 packages in 2m 11s
Uninstalled 9 packages in 1.94s
Installed 84 packages in 750ms
 + adagio==0.2.6
 + aiohttp-cors==0.8.1
 + alembic==1.18.5
 + autogluon==1.5.0
 + autogluon-common==1.5.0
 + autogluon-core==1.5.0
 + autogluon-features==1.5.0
 + autogluon-multimodal==1.5.0
 + autogluon-tabular==1.5.0
 + autogluon-timeseries==1.5.0
 + boto3==1.43.39
 + botocore==1.43.39
 + catboost==1.2.10
 + chronos-forecasting==2.3.0
 + colorama==0.4.6
 + colorful==0.5.8
 + colorlog==6.10.1
 + coreforecast==0.0.16
 + distlib==0.4.3
 + einx==0.4.3
 + evaluate==0.4.6
 + fugue==0.9.7
 + gluonts==0.16.3
 - huggingface-hub==1.20.1
 + huggingface-hub==0.36.2
 + jmespath==1.1.0
 - jsonschema==4.26.0
 + jsonschema==4.23.0
 + lightning==2.5.6
 + lightning-utilities==0.15.3
 + loguru==0.7.3
 + mako==1.3.12
 + mlforecast==0.14.0
 + model-index==0.1.11


In [5]:
# funcion para hacer submits a Kaggle
def kaggle_submit(competencia, archivo, mensaje):

  # comando
  comando = f'kaggle competitions submit -c {competencia} -f {archivo} -m "{mensaje}"'
  # ejecucion
  os.system(comando)


In [6]:
import os as os
import numpy as np
import polars as pl

from datetime import datetime
from autogluon.timeseries import TimeSeriesPredictor, TimeSeriesDataFrame

Por favor, cargar aqui SU semilla primigenia
<br> **Muy importante**, cambiar el numero de experimento en cada corrida. Usted ha sido notificado !
<br> Si cada corrida no está en una nueva carpeta virgen, entonces se reutilizarn modelos viejos corridos con otros parametros
https://www.youtube.com/shorts/0_0Kzqpdn1o

In [7]:
# defino los parametros
PARAM = {'experimento':'AutoGluon-01',
  'kaggle_competition':'labo-iii-2026-ba',
  'semilla_primigenia':109099
}

In [8]:
# creo la carpeta del experimento y hago el chdir
ruta = "/content/buckets/b1/exp/" + PARAM['experimento']
print(ruta)
os.makedirs(ruta, exist_ok=True)
os.chdir(ruta)

/content/buckets/b1/exp/AutoGluon-01


## 1.2 Init AutoGluon

In [9]:
# cargo el dataset del sell-in
dataset = pl.read_csv('/content/.drive/My Drive/labo3/datasets/sell-in.txt.gz', separator="\t")

In [10]:
# agrupo por product_id, periodo
tb_ventas = dataset.group_by("product_id", "periodo").agg(
    pl.col("tn").sum().alias("tn")
)

tb_ventas = tb_ventas.sort(["product_id", "periodo"])

In [11]:
# cargo la tabla "apredecir" que contiene los 780 productos que deben predecirse las ventas de 202002
tb_apredecir = pl.read_csv('/content/.drive/My Drive/labo3/datasets/product_id_apredecir201912.txt', separator="\t")


# Filtro tb_ventas a solo las que debo predecir
print(tb_ventas.height)
tb_ventas = tb_ventas.join(tb_apredecir,
  on="product_id",
  how="inner"
)
print(tb_ventas.height)
tb_ventas = tb_ventas.sort(["product_id", "periodo"])

31243
22349


In [12]:
# paso de periodo a  timestamp
tb_ventas = tb_ventas.with_columns(
    (pl.col('periodo').cast(pl.String).str.to_datetime('%Y%m')).alias('timestamp')
)

Opcion de *empiojar el dataset*
<br>agregando ruido relativo a las ventas
<br>Un Experimento no se le niega a nadie

In [13]:
empiojar= False
empiojar_ruido= 0.25

if empiojar:
  np.random.seed(PARAM['semilla_primigenia'])
  tb_ventas = tb_ventas.sort(["product_id", "periodo"])
  # vector con el ruido multiplicativo de media 1.0  y desvio  'empiojar_ruido'
  noise_multiplier = np.random.lognormal(mean=0.0, sigma=empiojar_ruido, size=tb_ventas.height)

  tb_ventas = tb_ventas.with_columns(
    (pl.col("tn") * pl.lit(noise_multiplier)).alias("tn")
  )


## 1.3 Entrenamiento AutoGluon

La magia del Auto Machine Learning  aplicada a un dataset que posee MULTIPLES series de tiempo que se analizan en forma conjunta, suponiendo algun tipo de correlacion entre ellas.

AutoGluon TimeSeriesPredictor predicts future values of **multiple related** time series.

TimeSeriesPredictor provides probabilistic (quantile) multi-step-ahead orecasts for univariate time series. The forecast includes both the mean (i.e.,
 onditional expectation of future values given the past), as well as the quantiles of the forecast distribution, indicating the range of possible future outcomes.

TimeSeriesPredictor fits both “global” deep learning models that are **shared across all time series** (e.g., DeepAR, Transformer), as well as “local”
 statistical models that are fit to each individual time series (e.g., ARIMA, ETS).

TimeSeriesPredictor expects input data and makes predictions in the TimeSeriesDataFrame format.


https://auto.gluon.ai/stable/api/autogluon.timeseries.TimeSeriesPredictor.html

In [14]:
# paso a formato  ts = time_series
ts_data = TimeSeriesDataFrame.from_data_frame(
  tb_ventas.to_pandas(), # tristemente AutoGluon espera un dataframe Pandas ...
  timestamp_column='timestamp',
  id_column='product_id'
)

Elegir cuidadosamente la metrica a utilizar
<br>Probar alternativas !
<br>Ese celda lleva 30 minutos en correr
<br>https://auto.gluon.ai/stable/tutorials/timeseries/forecasting-metrics.html#forecasting-metrics
<br>https://auto.gluon.ai/stable/api/autogluon.timeseries.TimeSeriesPredictor.fit.html#autogluon.timeseries.TimeSeriesPredictor.fit
<br>https://auto.gluon.ai/stable/api/autogluon.timeseries.TimeSeriesPredictor.html

In [15]:
# Entrenamiento, esta es la parte pesada
global_eval_metric = 'RMSE' # alternativa RMSE

# defino
modelo = TimeSeriesPredictor(
  prediction_length= 2,  # horizonte de prediccion
  target= 'tn',
  freq= 'MS',  # Frecuencia mensual (Month Start)
  eval_metric= global_eval_metric
)

# entreno, le tira con muchisimos algorimtos, y prueba ensamblarlos
modelo.fit(ts_data,
  num_val_windows= 2,
  time_limit= 3600, # una hora
  presets= "best_quality",  # Máxima calidad posible, obvio mayor tiempo de corrida
  random_seed= PARAM['semilla_primigenia']
)

Beginning AutoGluon training... Time limit = 3600s
AutoGluon will save models to '/content/.drive/MyDrive/labo 3/exp/AutoGluon-01/AutogluonModels/ag-20260701_224559'
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.13
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Thu Apr 30 18:17:14 UTC 2026
CPU Count:          2
Pytorch Version:    2.9.1+cu128
CUDA Version:       CUDA is not available
GPU Memory:         
Total GPU Memory:   Free: 0.00 GB, Allocated: 0.00 GB, Total: 0.00 GB
GPU Count:          0
Memory Avail:       11.08 GB / 12.67 GB (87.4%)
Disk Space Avail:   76.12 GB / 107.72 GB (70.7%)
Setting presets to: best_quality

Fitting with arguments:
{'enable_ensemble': True,
 'eval_metric': RMSE,
 'freq': 'MS',
 'hyperparameters': 'default',
 'known_covariates_names': [],
 'num_val_windows': 2,
 'prediction_length': 2,
 'quantile_levels': [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9],
 'random_seed':

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/478M [00:00<?, ?B/s]

	-33.2467      = Validation score (-RMSE)
	61.26   s     = Training runtime
	38.04   s     = Validation (prediction) runtime
Training timeseries model Chronos2SmallFineTuned. Training for up to 575.9s of the 3455.3s of remaining time.


config.json:   0%|          | 0.00/969 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/112M [00:00<?, ?B/s]

	-33.2152      = Validation score (-RMSE)
	549.08  s     = Training runtime
	9.42    s     = Validation (prediction) runtime
Training timeseries model AutoETS. Training for up to 579.3s of the 2896.7s of remaining time.
	-36.9451      = Validation score (-RMSE)
	24.23   s     = Training runtime
	25.63   s     = Validation (prediction) runtime
Training timeseries model ChronosWithRegressor[bolt_small]. Training for up to 748.9s of the 2846.8s of remaining time.
	Skipping covariate_regressor since the dataset contains no known_covariates or static_features.


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/191M [00:00<?, ?B/s]

	Skipping covariate_regressor since the dataset contains no known_covariates or static_features.
	-34.7490      = Validation score (-RMSE)
	5.74    s     = Training runtime
	3.78    s     = Validation (prediction) runtime
Training timeseries model TemporalFusionTransformer. Training for up to 1118.6s of the 2837.2s of remaining time.
	-34.9124      = Validation score (-RMSE)
	460.25  s     = Training runtime
	0.74    s     = Validation (prediction) runtime
Training timeseries model DeepAR. Training for up to 1776.1s of the 2376.1s of remaining time.
	-32.0990      = Validation score (-RMSE)
	177.02  s     = Training runtime
	2.47    s     = Validation (prediction) runtime
Fitting 1 ensemble(s), in 1 layers.
Training ensemble model WeightedEnsemble. Training for up to 2196.2s.
	Ensemble weights: {'AutoETS': 0.16, 'Chronos2': 0.08, 'Chronos2SmallFineTuned': 0.04, 'DeepAR': 0.21, 'SeasonalNaive': 0.38, 'TemporalFusionTransformer': 0.14}
	-28.7472      = Validation score (-RMSE)
	1.19    s

## 1.4 Prediccion AutoGluon

In [16]:
# predict a partir los mismos datos

tb_forecast = modelo.predict(ts_data,
  random_seed= PARAM['semilla_primigenia']
)

display(tb_forecast)

data with frequency 'IRREG' has been resampled to frequency 'MS'.
Model not specified in predict, will default to the model with the best validation score: WeightedEnsemble


mean         0.1          0.2          0.3  \
item_id timestamp                                                       
20001   2020-01-01  1361.753150  954.276208  1101.139673  1197.170297   
        2020-02-01  1371.572322  942.888602  1100.668566  1201.732679   
20002   2020-01-01  1234.776755  827.378141   973.845107  1071.386303   
        2020-02-01  1143.249908  729.020462   879.490427   984.508344   
20003   2020-01-01   850.958325  609.855000   694.234694   754.695254   
...                         ...         ...          ...          ...   
21266   2020-02-01     0.057331   -0.083114    -0.033733    -0.000156   
21267   2020-01-01     0.026566   -0.041950    -0.015258     0.000428   
        2020-02-01     0.031532   -0.059017    -0.026179    -0.005065   
21276   2020-01-01     0.011537   -0.012953    -0.003259     0.002476   
        2020-02-01     0.014580   -0.019223    -0.006390     0.001511   

                            0.4          0.5          0.6          0.7  \
item_id timestamp                                                        
20001   2020-01-01  1281.176884  1363.216561  1439.977300  1520.374211   
        2020-02-01  1280.828715  1368.777388  1450.087779  1538.022550   
20002   2020-01-01  1148.148020  1238.391508  1308.054514  1402.396212   
        2020-02-01  1064.376695  1150.907731  1223.758122  1317.399872   
20003   2020-01-01   803.496881   850.004281   895.691831   947.675384   
...                         ...          ...          ...          ...   
21266   2020-02-01     0.028483     0.057258     0.084356     0.116092   
21267   2020-01-01     0.013085     0.026569     0.039369     0.055322   
        2020-02-01     0.012697     0.031338     0.047795     0.069321   
21276   2020-01-01     0.006883     0.011628     0.016002     0.021598   
        2020-02-01     0.007496     0.014561     0.020423     0.028416   

                            0.8          0.9  
item_id timestamp                             
20001   2020-01-01  1634.952929  1790.656883  
        2020-02-01  1654.155434  1819.142927  
20002   2020-01-01  1504.897666  1662.475415  
        2020-02-01  1423.637668  1580.515124  
20003   2020-01-01  1013.059683  1107.673161  
...                         ...          ...  
21266   2020-02-01     0.157779     0.218394  
21267   2020-01-01     0.075722     0.104813  
        2020-02-01     0.096153     0.135722  
21276   2020-01-01     0.029165     0.039681  
        2020-02-01     0.038386     0.053862  

[1560 rows x 10 columns]

In [17]:
# paso a formato Polars, teniendo en cuenta el indice
tb_forecast = pl.from_pandas(tb_forecast.reset_index())

In [18]:
# me quedo con ls predicciones de febrero-2020
# en 'mean' esta la prediccion, mas alla de los n-tiles
tb_final = tb_forecast.filter(pl.col("timestamp") ==  datetime(2020, 2, 1)).select(["item_id","mean"])

display(tb_final)

item_id,mean
i64,f64
20001,1371.572322
20002,1143.249908
20003,728.215752
20004,530.523323
20005,534.33011
…,…
21263,0.039986
21265,0.05396
21266,0.057331


## 1.5 Submit a Kaggle

In [19]:
# cambio nombre de campos a los que reconoce Kaggle
tb_final = tb_final.rename({
  "item_id": "product_id",
  "mean": "tn",
})

In [20]:
# Submit a Kaggle
if not empiojar:
  archivo= "AutoGluon_" + global_eval_metric + ".csv"
  mensaje= "AutoGluon " + global_eval_metric
else:
  archivo= "AutoGluon_empiojado_" + global_eval_metric + ".csv"
  mensaje= "AutoGluon logEMPIOJADO " + global_eval_metric + " al " + str(empiojar_ruido)

tb_final.write_csv(archivo)

kaggle_submit(PARAM['kaggle_competition'], archivo, mensaje )